# B0 — Unguided SAM2 Baseline on Frozen FSC-147 dev100

## Purpose

Establish the **B0 control result on exactly the same frozen FSC-147 dev100 subset** used for the Group A–E development experiments and the subsequent end-to-end checkpoint.

### Experimental rule

- Reuse the frozen B0 SAM2 pipeline without optimisation or parameter tuning.
- Evaluate only the frozen **100-image FSC-147 validation subset**.
- Do not resample or redefine dev100.
- Do not access the FSC-147 test split.
- Preserve the original B0 SAM2 generation, mask-selection and counting rules.
- Add only the evaluation metrics and saved artefacts required for the end-to-end checkpoint.

### Evaluation

Report:

- MAE
- RMSE
- mean signed error / counting bias
- undercount / exact / overcount images
- mean and median relative absolute error
- exact-count rate
- percentage of images within 1%, 5%, 10% and 20% relative count error
- the above relevant metrics separately for S1–S6

### Output

Save the complete per-image B0-dev100 results and summary tables to the Google Drive project results directory.

This notebook establishes a **frozen control**.  
No B0 parameter is selected or tuned using dev100.

# 1. Setup

In [1]:
# 1.0 - mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 1.1 - Define Datasets paths

from pathlib import Path

# ============================================================
# Project / Dataset Paths
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

DATA_ROOT = PROJECT_ROOT / "datasets"


# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

FSC147_ROOT = DATA_ROOT / "FSC147"
FSC147_RAW = FSC147_ROOT / "raw"

FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
FSC147_DENSITY_DIR = FSC147_RAW / "gt_density_map_adaptive_384_VarV2"

FSC147_ANNOTATION_FILE = FSC147_RAW / "annotation_FSC147_384.json"
FSC147_SPLIT_FILE = FSC147_RAW / "Train_Test_Val_FSC_147.json"
FSC147_CLASSES_FILE = FSC147_RAW / "ImageClasses_FSC147.txt"


# ------------------------------------------------------------
# OmniCount-191
# ------------------------------------------------------------

OMNICOUNT_ROOT = DATA_ROOT / "OmniCount-191"


# ------------------------------------------------------------
# Quick confirmation
# ------------------------------------------------------------

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)

print("\nFSC-147:")
print("FSC147_ROOT  :", FSC147_ROOT)
print("FSC147_RAW   :", FSC147_RAW)

print("\nOmniCount-191:")
print("OMNICOUNT_ROOT:", OMNICOUNT_ROOT)



PROJECT_ROOT : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
DATA_ROOT    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets

FSC-147:
FSC147_ROOT  : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147
FSC147_RAW   : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147/raw

OmniCount-191:
OMNICOUNT_ROOT: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/OmniCount-191


In [3]:
# 1.2 Clone Respo

%cd /content
!rm -rf Surrey-MScProject-Zero-Shot-Object-Counting
!git clone https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git



/content
Cloning into 'Surrey-MScProject-Zero-Shot-Object-Counting'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 47 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 65.52 KiB | 8.19 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [4]:
# 1.3 Import Setup

import sys

repo_path = "/content/Surrey-MScProject-Zero-Shot-Object-Counting"

if repo_path not in sys.path:
    sys.path.append(repo_path)

from src.datasets import (
    FSC147Dataset,
    OmniCount191Dataset,
    collate_keep_dicts,
)

from src.evaluation import evaluate_single_label_counts

from torch.utils.data import DataLoader



# 2 - Load and Validate Frozen FSC-147 dev100

In [5]:
# ============================================================
# 2 - Load and Validate Frozen FSC-147 dev100
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Authoritative frozen dev100 definition
# ------------------------------------------------------------

DEV100_CSV = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "subsets"
    / "FSC147_val_dev_100.csv"
)

assert DEV100_CSV.exists(), (
    f"Frozen dev100 CSV not found:\n{DEV100_CSV}"
)

fsc147_dev_100_df = pd.read_csv(DEV100_CSV)


# ------------------------------------------------------------
# Basic frozen-subset checks
# ------------------------------------------------------------

required_cols = {
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
}

missing_cols = required_cols - set(
    fsc147_dev_100_df.columns
)

assert not missing_cols, (
    f"Missing required columns: {missing_cols}"
)

assert len(fsc147_dev_100_df) == 100

assert (
    fsc147_dev_100_df["dataset_index"]
    .nunique()
    == 100
)

assert (
    fsc147_dev_100_df["image_id"]
    .nunique()
    == 100
)


# ------------------------------------------------------------
# Load FSC-147 VALIDATION split only
# ------------------------------------------------------------

fsc147_val_dataset = FSC147Dataset(
    root=FSC147_RAW,
    split="val",
    load_images=False,
)

# ------------------------------------------------------------
# Validate every frozen row against the validation dataset
# ------------------------------------------------------------

validation_rows = []

for row in fsc147_dev_100_df.itertuples(index=False):

    dataset_index = int(row.dataset_index)

    assert 0 <= dataset_index < len(
        fsc147_val_dataset
    )

    sample = fsc147_val_dataset[
        dataset_index
    ]

    # The frozen image must be exactly the image stored at
    # this index in the FSC-147 validation dataset.
    assert (
        str(sample["image_id"])
        == str(row.image_id)
    ), (
        f"Image mismatch at dataset_index "
        f"{dataset_index}: "
        f'{sample["image_id"]} != {row.image_id}'
    )

    # FSC-147 dev100 is single-label.
    assert len(sample["categories"]) == 1

    category_name = next(
        iter(sample["categories"])
    )

    gt_count = int(
        sample["categories"][
            category_name
        ]["count"]
    )

    assert (
        str(category_name)
        == str(row.category)
    ), (
        f"Category mismatch for "
        f"{row.image_id}"
    )

    assert gt_count == int(
        row.gt_count
    ), (
        f"GT-count mismatch for "
        f"{row.image_id}"
    )

    validation_rows.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                row.image_id,

            "category":
                category_name,

            "gt_count":
                gt_count,

            "density_stratum":
                row.density_stratum,
        }
    )


dev100_validation_df = pd.DataFrame(
    validation_rows
)


# ------------------------------------------------------------
# Validate frozen density-stratum allocation
# ------------------------------------------------------------

expected_strata = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}

actual_strata = (
    fsc147_dev_100_df[
        "density_stratum"
    ]
    .value_counts()
    .to_dict()
)

assert actual_strata == expected_strata, (
    f"Unexpected density allocation:\n"
    f"{actual_strata}"
)


# ------------------------------------------------------------
# Final explicit experimental-universe statement
# ------------------------------------------------------------

print("=" * 90)
print("FROZEN FSC-147 DEV100 VALIDATION")
print("=" * 90)

print(f"Frozen CSV : {DEV100_CSV}")
print(f"FSC split  : val")
print(
    f"Val size   : "
    f"{len(fsc147_val_dataset)}"
)
print(
    f"Dev subset : "
    f"{len(fsc147_dev_100_df)}"
)

print("\nDensity strata:")

print(
    fsc147_dev_100_df[
        "density_stratum"
    ]
    .value_counts()
    .reindex(expected_strata.keys())
)

print(
    "\nAll dataset_index -> image_id "
    "mappings: PASS"
)

print(
    "All category mappings: PASS"
)

print(
    "All GT counts: PASS"
)

print(
    "Frozen S1-S6 allocation: PASS"
)

print(
    "\nTEST SET ACCESSED: NO"
)

print("=" * 90)

FROZEN FSC-147 DEV100 VALIDATION
Frozen CSV : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/subsets/FSC147_val_dev_100.csv
FSC split  : val
Val size   : 1286
Dev subset : 100

Density strata:
density_stratum
S1_0_20      20
S2_20_40     20
S3_40_60     20
S4_60_80     20
S5_80_95     10
S6_95_100    10
Name: count, dtype: int64

All dataset_index -> image_id mappings: PASS
All category mappings: PASS
All GT counts: PASS
Frozen S1-S6 allocation: PASS

TEST SET ACCESSED: NO


# 3. Unguided SAM2 automatic/grid generation

In [6]:
# ============================================================
# Step 3.1 - Initialise Unguided SAM2
# ============================================================

import torch
from pathlib import Path


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(
    "Device:",
    torch.cuda.get_device_name(0)
    if device.type == "cuda"
    else "CPU"
)


# ------------------------------------------------------------
# Load SAM 2.1 from Meta GitHub
# ------------------------------------------------------------

print("\nLoading SAM 2.1...")

%cd /content

if not Path("/content/sam2").exists():

    !git clone \
        https://github.com/facebookresearch/sam2.git


%cd /content/sam2

!pip install -q -e ".[notebooks]"


from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator


print("SAM 2.1 loaded.")


# ------------------------------------------------------------
# SAM2.1 Hiera-Small configuration
# ------------------------------------------------------------

SAM2_ROOT = Path("/content/sam2")

MODEL_CFG = (
    "configs/sam2.1/"
    "sam2.1_hiera_s.yaml"
)

CHECKPOINT = (
    SAM2_ROOT
    / "checkpoints"
    / "sam2.1_hiera_small.pt"
)


print(
    "Model config :",
    MODEL_CFG
)

print(
    "Checkpoint   :",
    CHECKPOINT
)


# ------------------------------------------------------------
# Download checkpoint if needed
# ------------------------------------------------------------

if not CHECKPOINT.exists():

    print(
        "Downloading SAM2.1 "
        "Hiera-Small checkpoint..."
    )

    CHECKPOINT.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    !wget -q \
        https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt \
        -O {CHECKPOINT}


assert CHECKPOINT.exists(), (
    f"Checkpoint not found: {CHECKPOINT}"
)

print(
    "Checkpoint exists:",
    CHECKPOINT.exists()
)


# ------------------------------------------------------------
# Build SAM2.1 Hiera-Small
# ------------------------------------------------------------

sam2_model = build_sam2(
    MODEL_CFG,
    str(CHECKPOINT),
    device=device
)


# Frozen B0 method:
# native SAM2AutomaticMaskGenerator,
# with no additional filtering.
mask_generator = SAM2AutomaticMaskGenerator(
    sam2_model
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print(
    "\nSAM2.1 Hiera-Small "
    "loaded successfully"
)

print(
    "Device         :",
    device
)

print(
    "Mask generator :",
    type(mask_generator).__name__
)

Device: Tesla T4

Loading SAM 2.1...
/content
Cloning into 'sam2'...
remote: Enumerating objects: 1107, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 1107 (delta 13), reused 7 (delta 7), pack-reused 1089 (from 2)
Receiving objects: 100% (1107/1107), 134.85 MiB | 19.71 MiB/s, done.
Resolving deltas: 100% (385/385), done.
/content/sam2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 428.6 kB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# ============================================================
# Code provenance - exact repository versions used
# ============================================================

import subprocess
import hashlib
import json
from pathlib import Path


PROJECT_CODE_ROOT = Path(
    "/content/Surrey-MScProject-Zero-Shot-Object-Counting"
)

SAM2_CODE_ROOT = Path(
    "/content/sam2"
)


def git_info(repo_path):

    repo_path = Path(repo_path)

    def git(*args):
        return subprocess.check_output(
            [
                "git",
                "-C",
                str(repo_path),
                *args,
            ],
            text=True,
        ).strip()

    return {
        "remote_origin":
            git("remote", "get-url", "origin"),

        "commit_sha":
            git("rev-parse", "HEAD"),

        "branch":
            git("rev-parse", "--abbrev-ref", "HEAD"),

        "dirty":
            bool(git("status", "--porcelain")),
    }


def sha256_file(path):

    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha256.update(block)

    return sha256.hexdigest()


code_provenance = {

    "project_repository":
        git_info(PROJECT_CODE_ROOT),

    "sam2_repository":
        git_info(SAM2_CODE_ROOT),

    "sam2_checkpoint": {
        "filename":
            CHECKPOINT.name,

        "sha256":
            sha256_file(CHECKPOINT),
    },

    "sam2_model_config":
        MODEL_CFG,
}


print("=" * 100)
print("CODE PROVENANCE")
print("=" * 100)

print(
    "\nProject repository:"
)
print(
    "  Commit:",
    code_provenance[
        "project_repository"
    ]["commit_sha"]
)
print(
    "  Branch:",
    code_provenance[
        "project_repository"
    ]["branch"]
)
print(
    "  Dirty:",
    code_provenance[
        "project_repository"
    ]["dirty"]
)

print(
    "\nSAM2 repository:"
)
print(
    "  Commit:",
    code_provenance[
        "sam2_repository"
    ]["commit_sha"]
)
print(
    "  Branch:",
    code_provenance[
        "sam2_repository"
    ]["branch"]
)
print(
    "  Dirty:",
    code_provenance[
        "sam2_repository"
    ]["dirty"]
)

print(
    "\nSAM2 checkpoint:",
    code_provenance[
        "sam2_checkpoint"
    ]["filename"]
)

print(
    "Checkpoint SHA256:",
    code_provenance[
        "sam2_checkpoint"
    ]["sha256"]
)

print(
    "\nModel config:",
    code_provenance[
        "sam2_model_config"
    ]
)

print("=" * 100)

CODE PROVENANCE

Project repository:
  Commit: ed9bce9b2c4825d67f800afe2fe0162d6e23fcb2
  Branch: main
  Dirty: True

SAM2 repository:
  Commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
  Branch: main
  Dirty: False

SAM2 checkpoint: sam2.1_hiera_small.pt
Checkpoint SHA256: 6d1aa6f30de5c92224f8172114de081d104bbd23dd9dc5c58996f0cad5dc4d38

Model config: configs/sam2.1/sam2.1_hiera_s.yaml


# 4. B0 Unguided SAM2 Mask Generation on Frozen dev100

In [8]:
# ============================================================
# Step 4 - B0 Unguided SAM2 Mask Generation on Frozen dev100
# ============================================================

import numpy as np
import time

start_time = time.time()

# ------------------------------------------------------------
# Experimental universe
# ------------------------------------------------------------

# IMPORTANT:
# - FSC-147 validation split only
# - exact frozen dev100 indices from Step 2
# - no resampling
# - no diagnostic/evaluation mode
# - native B0 SAM2 automatic mask generation unchanged

DEV100_INDICES = (
    fsc147_dev_100_df["dataset_index"]
    .astype(int)
    .tolist()
)

assert len(DEV100_INDICES) == 100
assert len(set(DEV100_INDICES)) == 100


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

b0_dataset = FSC147Dataset(
    root=FSC147_RAW,
    split="val",
    load_images=True
)

assert len(b0_dataset) == len(fsc147_val_dataset)


# ------------------------------------------------------------
# Run frozen B0 unguided native Meta SAM2
# ------------------------------------------------------------

raw_results = []

progress_total = len(DEV100_INDICES)

print("=" * 90)
print("B0 UNGUIDED SAM2 — FROZEN FSC-147 DEV100")
print("=" * 90)
print("Dataset : FSC-147")
print("Split   : val")
print(f"Images  : {progress_total}")
print("Method  : native SAM2 automatic mask generation")
print("Test set accessed: NO")
print("=" * 90)


for position, dataset_index in enumerate(DEV100_INDICES, start=1):

    sample = b0_dataset[dataset_index]
    image = sample["image"]

    # --------------------------------------------------------
    # Safety check against frozen dev100 definition
    # --------------------------------------------------------

    frozen_row = fsc147_dev_100_df.iloc[position - 1]

    assert int(frozen_row["dataset_index"]) == dataset_index

    assert (
        str(sample["image_id"])
        == str(frozen_row["image_id"])
    )

    assert sample["split"] == "val"


    print(
        f"[{position}/{progress_total}] "
        f"dataset_index={dataset_index} | "
        f"{sample['image_id']}"
    )


    # --------------------------------------------------------
    # Convert PIL image to HWC uint8 NumPy array
    # --------------------------------------------------------

    image_np = np.array(
        image.convert("RGB"),
        dtype=np.uint8,
        copy=True
    )


    # --------------------------------------------------------
    # Frozen B0 inference:
    # native SAM2 automatic mask generation
    # --------------------------------------------------------

    outputs = mask_generator.generate(image_np)

    num_raw_masks = len(outputs)


    # --------------------------------------------------------
    # Lightweight per-image result
    # --------------------------------------------------------

    category_name = next(iter(sample["categories"]))

    gt_count = int(
        sample["categories"][category_name]["count"]
    )

    result = {
        "dataset_index": dataset_index,
        "image_id": sample["image_id"],
        "image_path": sample["image_path"],
        "dataset": sample["dataset"],
        "split": sample["split"],
        "domain": sample["domain"],
        "width": sample["width"],
        "height": sample["height"],
        "category": category_name,
        "gt_count": gt_count,
        "density_stratum": frozen_row["density_stratum"],

        # Frozen B0 output
        "num_raw_masks": num_raw_masks,
    }

    raw_results.append(result)

    print(
        f"    GT: {gt_count} | "
        f"Generated masks: {num_raw_masks}"
    )


# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert len(raw_results) == 100

assert [
    str(r["image_id"])
    for r in raw_results
] == [
    str(x)
    for x in fsc147_dev_100_df["image_id"].tolist()
]

assert all(
    r["split"] == "val"
    for r in raw_results
)


elapsed_time = time.time() - start_time


print("\n" + "=" * 90)
print("B0 DEV100 INFERENCE COMPLETE")
print("=" * 90)

print(
    f"Images processed      : "
    f"{len(raw_results)}"
)

print(
    f"Total inference time  : "
    f"{elapsed_time / 60:.2f} minutes"
)

print(
    f"Average time / image  : "
    f"{elapsed_time / len(raw_results):.2f} seconds"
)

print("Frozen dev100 mapping : PASS")
print("Validation split only : PASS")
print("Test set accessed      : NO")

print("=" * 90)

B0 UNGUIDED SAM2 — FROZEN FSC-147 DEV100
Dataset : FSC-147
Split   : val
Images  : 100
Method  : native SAM2 automatic mask generation
Test set accessed: NO
[1/100] dataset_index=576 | 3873.jpg
    GT: 7 | Generated masks: 6
[2/100] dataset_index=322 | 1964.jpg
    GT: 8 | Generated masks: 7
[3/100] dataset_index=153 | 774.jpg
    GT: 8 | Generated masks: 5
[4/100] dataset_index=360 | 2304.jpg
    GT: 9 | Generated masks: 2
[5/100] dataset_index=514 | 3573.jpg
    GT: 9 | Generated masks: 7
[6/100] dataset_index=531 | 3661.jpg
    GT: 9 | Generated masks: 11
[7/100] dataset_index=880 | 5211.jpg
    GT: 9 | Generated masks: 12
[8/100] dataset_index=888 | 5220.jpg
    GT: 9 | Generated masks: 9
[9/100] dataset_index=112 | 623.jpg
    GT: 9 | Generated masks: 3
[10/100] dataset_index=119 | 737.jpg
    GT: 9 | Generated masks: 10
[11/100] dataset_index=170 | 824.jpg
    GT: 9 | Generated masks: 17
[12/100] dataset_index=559 | 3771.jpg
    GT: 10 | Generated masks: 15
[13/100] dataset_index

# 5 - B0 Mask selection

In [9]:
# ============================================================
# Step 5 - Mask selection / filtering
# B0: no additional filtering
# ============================================================

# Frozen B0 rule:
# Native SAM2 automatic masks are used directly.
# No additional semantic, geometric, size, score,
# overlap, or class-based filtering is applied.

for r in raw_results:
    r["num_selected_masks"] = r["num_raw_masks"]


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(raw_results) == 100

assert all(
    r["num_selected_masks"] == r["num_raw_masks"]
    for r in raw_results
)

print("=" * 80)
print("STEP 5 - B0 MASK SELECTION")
print("=" * 80)

print(f"Images processed : {len(raw_results)}")
print("Additional filtering applied : NO")
print("Selected masks = raw SAM2 masks: PASS")

print("=" * 80)

STEP 5 - B0 MASK SELECTION
Images processed : 100
Additional filtering applied : NO
Selected masks = raw SAM2 masks: PASS


In [10]:
# ============================================================
# Step 6 - Convert selected masks to predicted count
# ============================================================

# Frozen B0 counting rule:
# one selected SAM2 mask = one predicted object

for r in raw_results:
    r["pred_count"] = r["num_selected_masks"]


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(raw_results) == 100

assert all(
    r["pred_count"] == r["num_selected_masks"]
    for r in raw_results
)

assert all(
    "category" in r
    and "gt_count" in r
    and "density_stratum" in r
    for r in raw_results
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 90)
print("STEP 6 - B0 PREDICTED COUNTS")
print("=" * 90)

print(f"Images processed : {len(raw_results)}")
print("Predicted count = selected mask count: PASS")

print("\nFirst 5 results:")

for r in raw_results[:5]:
    print(
        f'{r["image_id"]} | '
        f'Target: {r["category"]} | '
        f'GT: {r["gt_count"]} | '
        f'Predicted: {r["pred_count"]} | '
        f'{r["density_stratum"]}'
    )

print("=" * 90)

STEP 6 - B0 PREDICTED COUNTS
Images processed : 100
Predicted count = selected mask count: PASS

First 5 results:
3873.jpg | Target: shallots | GT: 7 | Predicted: 6 | S1_0_20
1964.jpg | Target: bottle caps | GT: 8 | Predicted: 7 | S1_0_20
774.jpg | Target: books | GT: 8 | Predicted: 5 | S1_0_20
2304.jpg | Target: camels | GT: 9 | Predicted: 2 | S1_0_20
3573.jpg | Target: chicken wings | GT: 9 | Predicted: 7 | S1_0_20


In [11]:
# ============================================================
# Step 7 - Build dev100 per-image evaluation dataframe
# ============================================================

# One row per frozen dev100 image.
# No SAM2 inference is performed here.

evaluation_df = pd.DataFrame([
    {
        "dataset_index": r["dataset_index"],
        "image_id": r["image_id"],
        "category": r["category"],
        "gt_count": int(r["gt_count"]),
        "density_stratum": r["density_stratum"],
        "num_raw_masks": int(r["num_raw_masks"]),
        "num_selected_masks": int(r["num_selected_masks"]),
        "pred_count": int(r["pred_count"]),
    }
    for r in raw_results
])


# ------------------------------------------------------------
# Per-image error quantities
# ------------------------------------------------------------

evaluation_df["signed_error"] = (
    evaluation_df["pred_count"] - evaluation_df["gt_count"]
)

evaluation_df["abs_error"] = (
    evaluation_df["signed_error"].abs()
)

evaluation_df["squared_error"] = (
    evaluation_df["signed_error"] ** 2
)

# FSC-147 GT counts are positive, so this is well-defined.
evaluation_df["relative_error"] = (
    evaluation_df["abs_error"] / evaluation_df["gt_count"]
)


# ------------------------------------------------------------
# Error direction
# ------------------------------------------------------------

evaluation_df["error_direction"] = np.select(
    [
        evaluation_df["signed_error"] < 0,
        evaluation_df["signed_error"] == 0,
        evaluation_df["signed_error"] > 0,
    ],
    [
        "under",
        "exact",
        "over",
    ],
    default="unexpected",
)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert len(evaluation_df) == 100
assert evaluation_df["dataset_index"].is_unique
assert evaluation_df["image_id"].is_unique

assert (evaluation_df["gt_count"] > 0).all()

assert (
    evaluation_df["num_raw_masks"]
    == evaluation_df["num_selected_masks"]
).all()

assert (
    evaluation_df["num_selected_masks"]
    == evaluation_df["pred_count"]
).all()

assert set(evaluation_df["error_direction"].unique()).issubset(
    {"under", "exact", "over"}
)


# ------------------------------------------------------------
# Confirmation
# ------------------------------------------------------------

print("=" * 90)
print("STEP 7 - B0 DEV100 PER-IMAGE EVALUATION DATAFRAME")
print("=" * 90)

print(f"Rows                : {len(evaluation_df)}")
print(f"Unique dataset_index: {evaluation_df['dataset_index'].nunique()}")
print(f"Unique image_id     : {evaluation_df['image_id'].nunique()}")

print("\nColumns:")
print(list(evaluation_df.columns))

print("\nFirst 5 rows:")
display(evaluation_df.head())

print("=" * 90)

STEP 7 - B0 DEV100 PER-IMAGE EVALUATION DATAFRAME
Rows                : 100
Unique dataset_index: 100
Unique image_id     : 100

Columns:
['dataset_index', 'image_id', 'category', 'gt_count', 'density_stratum', 'num_raw_masks', 'num_selected_masks', 'pred_count', 'signed_error', 'abs_error', 'squared_error', 'relative_error', 'error_direction']

First 5 rows:


,dataset_index,image_id,category,gt_count,density_stratum,num_raw_masks,num_selected_masks,pred_count,signed_error,abs_error,squared_error,relative_error,error_direction
0,576,3873.jpg,shallots,7,S1_0_20,6,6,6,-1,1,1,0.142857,under
1,322,1964.jpg,bottle caps,8,S1_0_20,7,7,7,-1,1,1,0.125000,under
2,153,774.jpg,books,8,S1_0_20,5,5,5,-3,3,9,0.375000,under
3,360,2304.jpg,camels,9,S1_0_20,2,2,2,-7,7,49,0.777778,under
4,514,3573.jpg,chicken wings,9,S1_0_20,7,7,7,-2,2,4,0.222222,under


In [12]:
# ============================================================
# Step 8 - Overall B0-dev100 quantitative evaluation
# ============================================================

# All metrics are image-level metrics over the frozen
# 100-image FSC-147 validation subset.

n = len(evaluation_df)

metrics = {
    "num_samples": n,

    # Standard counting metrics
    "mae": evaluation_df["abs_error"].mean(),
    "rmse": np.sqrt(evaluation_df["squared_error"].mean()),
    "mean_signed_error": evaluation_df["signed_error"].mean(),

    # Error direction
    "num_undercount": int(
        (evaluation_df["error_direction"] == "under").sum()
    ),
    "num_exact": int(
        (evaluation_df["error_direction"] == "exact").sum()
    ),
    "num_overcount": int(
        (evaluation_df["error_direction"] == "over").sum()
    ),

    # Image-normalised relative-error metrics
    "mean_relative_abs_error": evaluation_df["relative_error"].mean(),
    "median_relative_abs_error": evaluation_df["relative_error"].median(),

    # Robustness / success rates
    "exact_count_rate": (
        (evaluation_df["signed_error"] == 0).mean() * 100
    ),
    "within_1pct": (
        (evaluation_df["relative_error"] <= 0.01).mean() * 100
    ),
    "within_5pct": (
        (evaluation_df["relative_error"] <= 0.05).mean() * 100
    ),
    "within_10pct": (
        (evaluation_df["relative_error"] <= 0.10).mean() * 100
    ),
    "within_20pct": (
        (evaluation_df["relative_error"] <= 0.20).mean() * 100
    ),
}


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert metrics["num_samples"] == 100

assert (
    metrics["num_undercount"]
    + metrics["num_exact"]
    + metrics["num_overcount"]
    == 100
)

assert metrics["mae"] >= 0
assert metrics["rmse"] >= 0

assert (
    0 <= metrics["exact_count_rate"] <= 100
    and 0 <= metrics["within_1pct"] <= 100
    and 0 <= metrics["within_5pct"] <= 100
    and 0 <= metrics["within_10pct"] <= 100
    and 0 <= metrics["within_20pct"] <= 100
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 90)
print("STEP 8 - B0 DEV100 OVERALL QUANTITATIVE EVALUATION")
print("=" * 90)

print(f"Number of images          : {metrics['num_samples']}")

print("\nCounting error")
print("-" * 45)
print(f"MAE                       : {metrics['mae']:.4f}")
print(f"RMSE                      : {metrics['rmse']:.4f}")
print(f"Mean signed error         : {metrics['mean_signed_error']:.4f}")

print("\nError direction")
print("-" * 45)
print(f"Undercount images         : {metrics['num_undercount']}")
print(f"Exact-count images        : {metrics['num_exact']}")
print(f"Overcount images          : {metrics['num_overcount']}")

print("\nImage-normalised error")
print("-" * 45)
print(
    f"Mean relative abs. error  : "
    f"{metrics['mean_relative_abs_error']:.4f} "
    f"({metrics['mean_relative_abs_error'] * 100:.2f}%)"
)
print(
    f"Median relative abs. error: "
    f"{metrics['median_relative_abs_error']:.4f} "
    f"({metrics['median_relative_abs_error'] * 100:.2f}%)"
)

print("\nRobustness rates")
print("-" * 45)
print(f"Exact-count rate          : {metrics['exact_count_rate']:.2f}%")
print(f"Within 1% relative error  : {metrics['within_1pct']:.2f}%")
print(f"Within 5% relative error  : {metrics['within_5pct']:.2f}%")
print(f"Within 10% relative error : {metrics['within_10pct']:.2f}%")
print(f"Within 20% relative error : {metrics['within_20pct']:.2f}%")

print("=" * 90)

STEP 8 - B0 DEV100 OVERALL QUANTITATIVE EVALUATION
Number of images          : 100

Counting error
---------------------------------------------
MAE                       : 80.6900
RMSE                      : 270.2339
Mean signed error         : -77.4700

Error direction
---------------------------------------------
Undercount images         : 76
Exact-count images        : 4
Overcount images          : 20

Image-normalised error
---------------------------------------------
Mean relative abs. error  : 0.5593 (55.93%)
Median relative abs. error: 0.6579 (65.79%)

Robustness rates
---------------------------------------------
Exact-count rate          : 4.00%
Within 1% relative error  : 4.00%
Within 5% relative error  : 7.00%
Within 10% relative error : 13.00%
Within 20% relative error : 22.00%


In [13]:
# ============================================================
# Step 9 - B0-dev100 metrics by density stratum (S1-S6)
# ============================================================

# Preserve the authoritative frozen dev100 stratum order.
stratum_order = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]

expected_stratum_sizes = {
    "S1_0_20": 20,
    "S2_20_40": 20,
    "S3_40_60": 20,
    "S4_60_80": 20,
    "S5_80_95": 10,
    "S6_95_100": 10,
}


# ------------------------------------------------------------
# Compute stratum-level metrics
# ------------------------------------------------------------

stratum_rows = []

for stratum in stratum_order:

    g = evaluation_df[
        evaluation_df["density_stratum"] == stratum
    ].copy()

    n = len(g)

    row = {
        "density_stratum": stratum,
        "num_images": n,

        # Standard counting metrics
        "mae": g["abs_error"].mean(),
        "rmse": np.sqrt(g["squared_error"].mean()),
        "mean_signed_error": g["signed_error"].mean(),

        # Error direction
        "num_undercount": int(
            (g["error_direction"] == "under").sum()
        ),
        "num_exact": int(
            (g["error_direction"] == "exact").sum()
        ),
        "num_overcount": int(
            (g["error_direction"] == "over").sum()
        ),

        # Image-normalised error
        "mean_relative_abs_error": g["relative_error"].mean(),
        "median_relative_abs_error": g["relative_error"].median(),

        # Robustness rates (%)
        "exact_count_rate": (
            (g["signed_error"] == 0).mean() * 100
        ),
        "within_1pct": (
            (g["relative_error"] <= 0.01).mean() * 100
        ),
        "within_5pct": (
            (g["relative_error"] <= 0.05).mean() * 100
        ),
        "within_10pct": (
            (g["relative_error"] <= 0.10).mean() * 100
        ),
        "within_20pct": (
            (g["relative_error"] <= 0.20).mean() * 100
        ),
    }

    stratum_rows.append(row)


stratum_metrics_df = pd.DataFrame(stratum_rows)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert list(
    stratum_metrics_df["density_stratum"]
) == stratum_order

assert (
    stratum_metrics_df["num_images"].sum()
    == 100
)

for stratum, expected_n in expected_stratum_sizes.items():

    actual_n = int(
        stratum_metrics_df.loc[
            stratum_metrics_df["density_stratum"] == stratum,
            "num_images"
        ].iloc[0]
    )

    assert actual_n == expected_n, (
        f"{stratum}: expected {expected_n}, got {actual_n}"
    )

assert (
    stratum_metrics_df[
        ["num_undercount", "num_exact", "num_overcount"]
    ].sum(axis=1)
    == stratum_metrics_df["num_images"]
).all()


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 110)
print("STEP 9 - B0 DEV100 PERFORMANCE BY DENSITY STRATUM")
print("=" * 110)

display(
    stratum_metrics_df[
        [
            "density_stratum",
            "num_images",
            "mae",
            "rmse",
            "mean_signed_error",
            "num_undercount",
            "num_exact",
            "num_overcount",
            "mean_relative_abs_error",
            "median_relative_abs_error",
            "exact_count_rate",
            "within_1pct",
            "within_5pct",
            "within_10pct",
            "within_20pct",
        ]
    ].round(4)
)

print("=" * 110)

STEP 9 - B0 DEV100 PERFORMANCE BY DENSITY STRATUM


,density_stratum,num_images,mae,rmse,mean_signed_error,num_undercount,num_exact,num_overcount,mean_relative_abs_error,median_relative_abs_error,exact_count_rate,within_1pct,within_5pct,within_10pct,within_20pct
0,S1_0_20,20,5.15,6.1115,-0.55,10,1,9,0.5049,0.5417,5.0,5.0,5.0,5.0,20.0
1,S2_20_40,20,6.55,8.1086,-5.55,16,2,2,0.4337,0.3000,10.0,10.0,10.0,20.0,30.0
2,S3_40_60,20,14.95,17.6423,-13.55,17,1,2,0.5531,0.5563,5.0,5.0,10.0,15.0,15.0
3,S4_60_80,20,34.30,38.6833,-29.10,16,0,4,0.6225,0.7910,0.0,0.0,10.0,10.0,20.0
4,S5_80_95,10,62.10,76.3001,-57.50,8,0,2,0.5256,0.6726,0.0,0.0,0.0,20.0,40.0
5,S6_95_100,10,622.90,848.8936,-619.70,9,0,1,0.8395,0.9868,0.0,0.0,0.0,10.0,10.0


In [14]:
# ============================================================
# Step 10 - B0-dev100 error concentration diagnostics
# ============================================================

# This quantifies how much each density stratum contributes
# to the total absolute error and total squared error.

total_abs_error = evaluation_df["abs_error"].sum()
total_squared_error = evaluation_df["squared_error"].sum()


# ------------------------------------------------------------
# Error contribution by density stratum
# ------------------------------------------------------------

error_concentration_df = (
    evaluation_df
    .groupby("density_stratum", sort=False)
    .agg(
        num_images=("image_id", "size"),
        total_abs_error=("abs_error", "sum"),
        total_squared_error=("squared_error", "sum"),
    )
    .reindex(stratum_order)
    .reset_index()
)

error_concentration_df["abs_error_share_pct"] = (
    error_concentration_df["total_abs_error"]
    / total_abs_error
    * 100
)

error_concentration_df["squared_error_share_pct"] = (
    error_concentration_df["total_squared_error"]
    / total_squared_error
    * 100
)


# ------------------------------------------------------------
# Worst individual images by absolute error
# ------------------------------------------------------------

worst_images_df = (
    evaluation_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
            "pred_count",
            "signed_error",
            "abs_error",
            "relative_error",
        ]
    ]
    .sort_values(
        ["abs_error", "dataset_index"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Top-k absolute-error concentration
# ------------------------------------------------------------

topk_rows = []

for k in [1, 5, 10, 20]:

    topk_abs_error = worst_images_df.head(k)["abs_error"].sum()

    topk_rows.append(
        {
            "top_k_images": k,
            "total_abs_error": topk_abs_error,
            "abs_error_share_pct": (
                topk_abs_error / total_abs_error * 100
            ),
        }
    )

topk_concentration_df = pd.DataFrame(topk_rows)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

assert error_concentration_df["num_images"].sum() == 100

assert (
    error_concentration_df["total_abs_error"].sum()
    == total_abs_error
)

assert (
    error_concentration_df["total_squared_error"].sum()
    == total_squared_error
)

assert np.isclose(
    error_concentration_df["abs_error_share_pct"].sum(),
    100.0
)

assert np.isclose(
    error_concentration_df["squared_error_share_pct"].sum(),
    100.0
)


# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("=" * 100)
print("STEP 10 - B0 DEV100 ERROR CONCENTRATION")
print("=" * 100)

print("\nError contribution by density stratum:")
display(
    error_concentration_df.round(4)
)

print("\nTop-k image contribution to total absolute error:")
display(
    topk_concentration_df.round(4)
)

print("\n10 worst individual images by absolute error:")
display(
    worst_images_df.head(10).round(4)
)

print("=" * 100)

STEP 10 - B0 DEV100 ERROR CONCENTRATION

Error contribution by density stratum:


,density_stratum,num_images,total_abs_error,total_squared_error,abs_error_share_pct,squared_error_share_pct
0,S1_0_20,20,103,747,1.2765,0.0102
1,S2_20_40,20,131,1315,1.6235,0.0180
2,S3_40_60,20,299,6225,3.7055,0.0852
3,S4_60_80,20,686,29928,8.5017,0.4098
4,S5_80_95,10,621,58217,7.6961,0.7972
5,S6_95_100,10,6229,7206203,77.1967,98.6795



Top-k image contribution to total absolute error:


,top_k_images,total_abs_error,abs_error_share_pct
0,1,2087,25.8644
1,5,5277,65.3984
2,10,6340,78.5723
3,20,7057,87.4582



10 worst individual images by absolute error:


,dataset_index,image_id,category,density_stratum,gt_count,pred_count,signed_error,abs_error,relative_error
0,229,935.jpg,birds,S6_95_100,2092,5,-2087,2087,0.9976
1,1192,6969.jpg,birds,S6_95_100,949,0,-949,949,1.0000
2,535,3665.jpg,toilet paper rolls,S6_95_100,907,14,-893,893,0.9846
3,253,975.jpg,birds,S6_95_100,718,0,-718,718,1.0000
4,184,840.jpg,books,S6_95_100,637,7,-630,630,0.9890
5,1187,6931.jpg,birds,S6_95_100,355,3,-352,352,0.9915
6,938,5866.jpg,grapes,S6_95_100,267,47,-220,220,0.8240
7,438,3426.jpg,polka dots,S6_95_100,219,11,-208,208,0.9498
8,9,215.jpg,grapes,S6_95_100,259,103,-156,156,0.6023
9,298,1935.jpg,bottle caps,S5_80_95,175,48,-127,127,0.7257


In [17]:
# ============================================================
# Step 11 - Freeze B0-dev100 artefacts to Google Drive
# ============================================================

import json


# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

B0_DEV100_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "B0_dev100"
)

B0_DEV100_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Overall summary table
# ------------------------------------------------------------

overall_metrics_df = pd.DataFrame(
    [metrics]
)


# ------------------------------------------------------------
# Frozen protocol metadata
# ------------------------------------------------------------

b0_dev100_metadata = {
    "stage": "B0-dev100",
    "dataset": "FSC-147",
    "split": "val",
    "num_images": int(len(evaluation_df)),
    "dev_subset_csv": str(DEV100_CSV),

    "model": "SAM2.1 Hiera-Small",
    "model_config": MODEL_CFG,
    "checkpoint": str(CHECKPOINT),
    "mask_generator": type(mask_generator).__name__,

    "code_provenance": code_provenance,

    "prompting_mode": "unguided automatic mask generation",
    "additional_mask_filtering": False,
    "counting_rule": "pred_count = num_selected_masks = num_raw_masks",

    "total_inference_seconds": float(elapsed_time),
    "mean_inference_seconds_per_image": float(
        elapsed_time / len(raw_results)
    ),

    "test_set_accessed": False,
    "purpose": (
        "Frozen apples-to-apples B0 control for the "
        "dev100 end-to-end semantic-guided SAM2 checkpoint"
    ),
}


# ------------------------------------------------------------
# Files to save
# ------------------------------------------------------------

save_tables = {
    "B0_dev100_per_image.csv":
        evaluation_df,

    "B0_dev100_overall_metrics.csv":
        overall_metrics_df,

    "B0_dev100_density_metrics.csv":
        stratum_metrics_df,

    "B0_dev100_error_concentration_by_density.csv":
        error_concentration_df,

    "B0_dev100_topk_error_concentration.csv":
        topk_concentration_df,

    "B0_dev100_worst_images.csv":
        worst_images_df,
}


# ------------------------------------------------------------
# Save CSV tables
# ------------------------------------------------------------

for filename, df in save_tables.items():

    path = B0_DEV100_RESULTS_DIR / filename

    df.to_csv(
        path,
        index=False
    )

    print(
        f"SAVED: {path} "
        f"{df.shape}"
    )


# ------------------------------------------------------------
# Save metadata JSON
# ------------------------------------------------------------

metadata_path = (
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_metadata.json"
)

with open(
    metadata_path,
    "w"
) as f:

    json.dump(
        b0_dev100_metadata,
        f,
        indent=2
    )

print(
    f"SAVED: {metadata_path}"
)


# ------------------------------------------------------------
# Round-trip validation
# ------------------------------------------------------------

for filename, original_df in save_tables.items():

    path = B0_DEV100_RESULTS_DIR / filename

    assert path.exists()

    reloaded_df = pd.read_csv(path)

    assert reloaded_df.shape == original_df.shape, (
        f"Shape mismatch after reload: {filename}"
    )


assert metadata_path.exists()

with open(
    metadata_path,
    "r"
) as f:

    reloaded_metadata = json.load(f)

assert reloaded_metadata["num_images"] == 100
assert reloaded_metadata["split"] == "val"
assert reloaded_metadata["test_set_accessed"] is False
assert reloaded_metadata["additional_mask_filtering"] is False


# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 100)
print("STEP 11 - B0 DEV100 ARTEFACT FREEZE COMPLETE")
print("=" * 100)

print(
    "Results directory :",
    B0_DEV100_RESULTS_DIR
)

print(
    "Saved tables      :",
    len(save_tables)
)

print(
    "Saved metadata    :",
    metadata_path.name
)

print(
    "Round-trip checks : PASS"
)

print(
    "Validation only   : PASS"
)

print(
    "Test set accessed : NO"
)

print("=" * 100)

SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_per_image.csv (100, 13)
SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_overall_metrics.csv (1, 14)
SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_density_metrics.csv (6, 15)
SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_error_concentration_by_density.csv (6, 6)
SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_topk_error_concentration.csv (4, 3)
SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_worst_images.csv (100, 9)
SAVED: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100/B0_dev100_metadata.json

STEP 11 - B0 DEV100 ARTEFACT FREEZE COMPLETE
Results directory : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100
Saved tables      : 6
Saved metadata    : B0_dev1

In [19]:
# ============================================================
# Step 12 - Final B0-dev100 freeze validation
# ============================================================

# Reload the authoritative saved per-image B0 result.
saved_per_image_df = pd.read_csv(
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_per_image.csv"
)

saved_overall_df = pd.read_csv(
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_overall_metrics.csv"
)

saved_density_df = pd.read_csv(
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_density_metrics.csv"
)


# ------------------------------------------------------------
# 1. Basic saved-table integrity
# ------------------------------------------------------------

assert len(saved_per_image_df) == 100
assert saved_per_image_df["dataset_index"].is_unique
assert saved_per_image_df["image_id"].is_unique


# ------------------------------------------------------------
# 2. Exact dev100 ordering and mapping
# ------------------------------------------------------------

assert (
    saved_per_image_df["dataset_index"].astype(int).tolist()
    ==
    fsc147_dev_100_df["dataset_index"].astype(int).tolist()
)

assert (
    saved_per_image_df["image_id"].astype(str).tolist()
    ==
    fsc147_dev_100_df["image_id"].astype(str).tolist()
)

assert (
    saved_per_image_df["category"].astype(str).tolist()
    ==
    fsc147_dev_100_df["category"].astype(str).tolist()
)

assert (
    saved_per_image_df["gt_count"].astype(int).tolist()
    ==
    fsc147_dev_100_df["gt_count"].astype(int).tolist()
)

assert (
    saved_per_image_df["density_stratum"].astype(str).tolist()
    ==
    fsc147_dev_100_df["density_stratum"].astype(str).tolist()
)


# ------------------------------------------------------------
# 3. Exact frozen B0 counting rule
# ------------------------------------------------------------

assert (
    saved_per_image_df["num_raw_masks"]
    ==
    saved_per_image_df["num_selected_masks"]
).all()

assert (
    saved_per_image_df["num_selected_masks"]
    ==
    saved_per_image_df["pred_count"]
).all()


# ------------------------------------------------------------
# 4. Saved per-image values reproduce in-memory results
# ------------------------------------------------------------

key_cols = [
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
    "num_raw_masks",
    "num_selected_masks",
    "pred_count",
]

for col in key_cols:

    if col in ["image_id", "category", "density_stratum"]:

        assert (
            saved_per_image_df[col].astype(str).tolist()
            ==
            evaluation_df[col].astype(str).tolist()
        )

    else:

        assert (
            saved_per_image_df[col].tolist()
            ==
            evaluation_df[col].tolist()
        )


for col in [
    "signed_error",
    "abs_error",
    "squared_error",
    "relative_error",
]:

    assert np.allclose(
        saved_per_image_df[col].to_numpy(),
        evaluation_df[col].to_numpy(),
    )


assert (
    saved_per_image_df["error_direction"].astype(str).tolist()
    ==
    evaluation_df["error_direction"].astype(str).tolist()
)


# ------------------------------------------------------------
# 5. Overall saved metrics reproduce frozen metrics
# ------------------------------------------------------------

assert len(saved_overall_df) == 1

for metric_name, metric_value in metrics.items():

    assert metric_name in saved_overall_df.columns

    assert np.isclose(
        float(saved_overall_df.iloc[0][metric_name]),
        float(metric_value),
    )


# ------------------------------------------------------------
# 6. Density table integrity
# ------------------------------------------------------------

assert (
    saved_density_df["density_stratum"].astype(str).tolist()
    ==
    stratum_order
)

assert (
    saved_density_df["num_images"].astype(int).tolist()
    ==
    [20, 20, 20, 20, 10, 10]
)

assert saved_density_df["num_images"].sum() == 100


# ------------------------------------------------------------
# 7. Metadata integrity
# ------------------------------------------------------------

with open(
    B0_DEV100_RESULTS_DIR
    / "B0_dev100_metadata.json",
    "r"
) as f:

    final_metadata = json.load(f)

assert final_metadata["dataset"] == "FSC-147"
assert final_metadata["split"] == "val"
assert final_metadata["num_images"] == 100
assert final_metadata["additional_mask_filtering"] is False
assert final_metadata["test_set_accessed"] is False

assert (
    final_metadata["counting_rule"]
    ==
    "pred_count = num_selected_masks = num_raw_masks"
)


# ------------------------------------------------------------
# Provenance integrity
# ------------------------------------------------------------

assert "code_provenance" in final_metadata

assert (
    "sam2_repository"
    in final_metadata["code_provenance"]
)

assert (
    "sam2_checkpoint"
    in final_metadata["code_provenance"]
)

assert (
    final_metadata[
        "code_provenance"
    ]["sam2_repository"]["commit_sha"]
    ==
    "2b90b9f5ceec907a1c18123530e92e794ad901a4"
)


# ------------------------------------------------------------
# Final confirmation
# ------------------------------------------------------------

print("=" * 100)
print("STEP 12 - B0 DEV100 FINAL FREEZE VALIDATION")
print("=" * 100)

print("Saved per-image rows             : 100")
print("Authoritative dev100 order       : PASS")
print("dataset_index mapping            : PASS")
print("image_id mapping                 : PASS")
print("category mapping                 : PASS")
print("GT count mapping                 : PASS")
print("density-stratum mapping          : PASS")
print("Frozen B0 counting rule          : PASS")
print("Saved per-image values           : PASS")
print("Saved overall metrics            : PASS")
print("Saved S1-S6 summary              : PASS")
print("Metadata                         : PASS")
print("Validation split only            : PASS")
print("Test set accessed                : NO")

print("\nB0-dev100 frozen control:")
print(B0_DEV100_RESULTS_DIR)

print("=" * 100)

STEP 12 - B0 DEV100 FINAL FREEZE VALIDATION
Saved per-image rows             : 100
Authoritative dev100 order       : PASS
dataset_index mapping            : PASS
image_id mapping                 : PASS
category mapping                 : PASS
GT count mapping                 : PASS
density-stratum mapping          : PASS
Frozen B0 counting rule          : PASS
Saved per-image values           : PASS
Saved overall metrics            : PASS
Saved S1-S6 summary              : PASS
Metadata                         : PASS
Validation split only            : PASS
Test set accessed                : NO

B0-dev100 frozen control:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B0_dev100
